In [8]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.styles import numbers
from datetime import datetime
import glob

# Step 1: Automatically detect the CSV file that starts with "unmatched_export" in the current directory
file_pattern = "unmatched_export*.csv"
input_files = glob.glob(file_pattern)

if not input_files:
    raise FileNotFoundError(f"No files matching '{file_pattern}' found in the current directory.")
else:
    input_filename = input_files[0]  # Take the first matching file

# Step 2: Detect the VM Analysis file
vm_analysis_pattern = "*VM Analysis*.xlsx"
vm_files = glob.glob(vm_analysis_pattern)

if not vm_files:
    raise FileNotFoundError(f"No files matching '{vm_analysis_pattern}' found in the current directory.")
else:
    vm_analysis_filename = vm_files[0]

# Step 3: Prepare the output filename
today = datetime.today().strftime('%Y-%m-%d')
output_filename = f"Step 5_SDP Unmatched Refined - {today}.xlsx"

# Load the VM Analysis file and extract Spend Summary tab
df_vm = pd.read_excel(vm_analysis_filename, sheet_name='Spend Summary')
df_vm = df_vm[['internal supplier id', 'aggregated spend']]  # Ensure relevant columns are loaded

# Load the unmatched CSV file
df_unmatched = pd.read_csv(input_filename)

# Step 4: Add "aggregated spend" column to the Unmatched Data tab
df_unmatched = pd.merge(df_unmatched, df_vm, how='left', left_on='internal_supplier_id', right_on='internal supplier id')
df_unmatched = df_unmatched.drop(columns=['internal supplier id'])  # Clean up
df_unmatched['aggregated spend'] = df_unmatched['aggregated spend'].fillna(0)  # Handle missing spend values

# Move the "aggregated spend" column to appear after "complete_address" and before "unmatched_reason"
cols = list(df_unmatched.columns)
aggregated_spend_idx = cols.index('aggregated spend')
complete_address_idx = cols.index('complete_address')
# Reorder columns
cols.insert(complete_address_idx + 1, cols.pop(aggregated_spend_idx))
df_unmatched = df_unmatched[cols]

# After processing, restrict df_unmatched to the required columns
columns_to_keep = ['internal_supplier_id', 'company_name', 'complete_address', 'aggregated spend', 'unmatched_reason', 'error_1', 'error_2']
df_unmatched = df_unmatched[columns_to_keep]

# Define color fills, fonts, and alignment
teal_fill = PatternFill(start_color="008080", end_color="008080", fill_type="solid")
bold_font = Font(bold=True)
white_font = Font(color="FFFFFF", bold=True)
right_align = Alignment(horizontal="right")
aqua_blue_fill = PatternFill(start_color="00FFFF", end_color="00FFFF", fill_type="solid")
light_aqua_blue_fill = PatternFill(start_color="E0FFFF", end_color="E0FFFF", fill_type="solid")
bright_green_fill = PatternFill(start_color="00FF00", end_color="00FF00", fill_type="solid")
lighter_bright_green_fill = PatternFill(start_color="CCFFCC", end_color="CCFFCC", fill_type="solid")  # Lighter green
red_fill = PatternFill(start_color="FF0000", end_color="FF0000", fill_type="solid")
light_red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
yellow_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
thin = Side(border_style="thin", color="000000")

# Step 5: Create a new Excel writer object
with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    # Write the modified unmatched data to the "Unmatched Data" sheet
    df_unmatched.to_excel(writer, sheet_name='Unmatched Data', index=False)
    
    # Apply conditional formatting in the "Unmatched Data" sheet
    workbook = writer.book
    ws_data = workbook['Unmatched Data']

    # Apply teal fill and bold font to headers in the "Unmatched Data" tab
    for cell in ws_data[1]:
        cell.fill = teal_fill
        cell.font = white_font

    # Set column width for relevant columns that exist in the DataFrame
    columns_to_resize = ["company_name", "complete_address", "aggregated spend", "unmatched_reason", "error_1", "error_2"]
    for col in columns_to_resize:
        if col in df_unmatched.columns:  # Check if the column exists before attempting to resize
            col_idx = df_unmatched.columns.get_loc(col) + 1  # Get the index of the column (1-based for openpyxl)
            ws_data.column_dimensions[get_column_letter(col_idx)].width = 35  # Set width to 35

    # Apply conditional formatting for "unmatched_reason" column
    unmatched_reason_idx = df_unmatched.columns.get_loc('unmatched_reason') + 1  # Get index (1-based for openpyxl)

    for row in ws_data.iter_rows(min_row=2, max_row=ws_data.max_row, min_col=unmatched_reason_idx, max_col=unmatched_reason_idx):
        cell = row[0]
        if "Record Dropped" in str(cell.value):
            cell.fill = aqua_blue_fill
        elif "No Match Found" in str(cell.value):
            cell.fill = red_fill
        elif "Record Failed" in str(cell.value):
            cell.fill = yellow_fill
        elif "Duplicate Removed" in str(cell.value):
            cell.fill = bright_green_fill

    # Apply light aqua blue highlight to the "aggregated spend" column
    aggregated_spend_idx = df_unmatched.columns.get_loc('aggregated spend') + 1  # Get index (1-based for openpyxl)
    for row in ws_data.iter_rows(min_row=2, max_row=ws_data.max_row, min_col=aggregated_spend_idx, max_col=aggregated_spend_idx):
        cell = row[0]
        cell.fill = light_aqua_blue_fill

    # Step 6: Create the "Unmatched Summary" sheet
    supplier_ids_count = df_unmatched['internal_supplier_id'].count()
    
    # Calculate Spend and Spend Percentage for each category
    total_spend = df_unmatched['aggregated spend'].sum()
    
    # Helper function to calculate the sum of spend for a given condition
    def calculate_spend(condition):
        return df_unmatched[condition]['aggregated spend'].sum()
    
    # Create the count list first
    count_list = [
        supplier_ids_count,
        df_unmatched[df_unmatched['unmatched_reason'].str.contains("Record Dropped", na=False)].shape[0],
        df_unmatched[df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('supplier missing minimum required fields: supplier_name and address', na=False)).any(axis=1)].shape[0],
        df_unmatched[df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('Record must have either a complete address or a combination of street address, city, and country', na=False)).any(axis=1)].shape[0],
        df_unmatched[df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('the postal code is incorrect for the specified country/state', na=False)).any(axis=1)].shape[0],
        df_unmatched[df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('Record must have either a complete address or individual fields', na=False)).any(axis=1)].shape[0],
        df_unmatched[df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('missing values in the mandatory Internal Supplier Id', na=False)).any(axis=1)].shape[0],
        df_unmatched[df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('supplier name has more than 50% non-English characters', na=False)).any(axis=1)].shape[0],
        df_unmatched[df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('missing values in the mandatory Company Name', na=False)).any(axis=1)].shape[0],
        df_unmatched['error_2'].notna().sum(),
        df_unmatched[df_unmatched['unmatched_reason'].str.contains("No Match Found", na=False)].shape[0],
        df_unmatched[df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('We did not find a match for this record in our database', na=False)).any(axis=1)].shape[0],
        df_unmatched[df_unmatched['unmatched_reason'].str.contains("Record Failed", na=False)].shape[0],
        df_unmatched[df_unmatched['unmatched_reason'].str.contains("Duplicate Removed", na=False)].shape[0],
        df_unmatched[df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('This record was an exact duplicate of another record', na=False)).any(axis=1)].shape[0]
    ]

    # Now create the summary_data dictionary without "Percentage" and "Spend Percentage"
    summary_data = {
        "Category": [
            "Supplier IDs",
            "Record Dropped",
            "Supplier missing minimum required fields: supplier_name and address",
            "Record must have either a complete address or a combination of street address, city, and country",
            "The postal code is incorrect for the specified country/state",
            "Record must have either a complete address or individual fields (like street address, city, etc.), but not both.",
            "Record is missing values in the mandatory Internal Supplier Id field.",
            "the supplier name has more than 50% non-English characters",
            "Record is missing values in the mandatory Company Name field.",
            "Multiple Errors",
            "No Match Found",
            "We did not find a match for this record in our database",
            "Record Failed",
            "Duplicate Removed",
            "This record was an exact duplicate of another record in your data and was therefore removed."
        ],
        "Count": count_list,
        "Spend": [
            df_unmatched['aggregated spend'].sum(),
            calculate_spend(df_unmatched['unmatched_reason'].str.contains("Record Dropped", na=False)),
            calculate_spend(df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('supplier missing minimum required fields: supplier_name and address', na=False)).any(axis=1)),
            calculate_spend(df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('Record must have either a complete address or a combination of street address, city, and country', na=False)).any(axis=1)),
            calculate_spend(df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('the postal code is incorrect for the specified country/state', na=False)).any(axis=1)),
            calculate_spend(df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('Record must have either a complete address or individual fields', na=False)).any(axis=1)),
            calculate_spend(df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('missing values in the mandatory Internal Supplier Id', na=False)).any(axis=1)),
            calculate_spend(df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('supplier name has more than 50% non-English characters', na=False)).any(axis=1)),
            calculate_spend(df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('missing values in the mandatory Company Name', na=False)).any(axis=1)),
            calculate_spend(df_unmatched['error_2'].notna()),
            calculate_spend(df_unmatched['unmatched_reason'].str.contains("No Match Found", na=False)),
            calculate_spend(df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('We did not find a match for this record in our database', na=False)).any(axis=1)),
            calculate_spend(df_unmatched['unmatched_reason'].str.contains("Record Failed", na=False)),
            calculate_spend(df_unmatched['unmatched_reason'].str.contains("Duplicate Removed", na=False)),
            calculate_spend(df_unmatched[['error_1', 'error_2']].apply(lambda x: x.str.contains('This record was an exact duplicate of another record', na=False)).any(axis=1))
        ]
    }
    
    # Now calculate "Percentage" and "Spend Percentage"
    summary_data["Percentage"] = [(count / supplier_ids_count) if supplier_ids_count > 0 else 0 for count in summary_data["Count"]]
    total_spend = summary_data["Spend"][0]  # Total spend for Supplier IDs
    summary_data["Spend Percentage"] = [(spend / total_spend) if total_spend > 0 else 0 for spend in summary_data["Spend"]]

    
    # Convert the summary data to a DataFrame and write it to the Unmatched Summary sheet
    summary_df = pd.DataFrame(summary_data)
    # Reorder the columns in the desired order
    summary_df = summary_df[['Category', 'Count', 'Percentage', 'Spend', 'Spend Percentage']]
    
    summary_df.to_excel(writer, sheet_name='Unmatched Summary', index=False)

    # Step 7: Apply formatting to the "Unmatched Summary" sheet
    ws_summary = workbook['Unmatched Summary']
    ws_summary.column_dimensions['A'].width = 74  # Set "Category" column width to 74
    ws_summary.column_dimensions['B'].width = 15  # Set width of the "Count" column to 15
    ws_summary.column_dimensions['C'].width = 15  # Set width of the "Percentage" column to 15
    ws_summary.column_dimensions['D'].width = 15  # Set width of the "Spend" column to 15
    ws_summary.column_dimensions['E'].width = 15  # Set width of the "Spend Percentage" column to 15
    
    # Apply teal fill and white font to headers in the "Unmatched Summary" tab
    for cell in ws_summary[1]:
        cell.fill = teal_fill
        cell.font = white_font
    
    # Apply right alignment to "Percentage", "Spend", and "Spend Percentage" columns
    for col in ['C', 'D', 'E']:
        for cell in ws_summary[col]:
            cell.alignment = right_align
    
    # Apply percentage format to the "Percentage" and "Spend Percentage" columns
    for row in ws_summary.iter_rows(min_row=2, max_row=ws_summary.max_row, min_col=3, max_col=3):
        for cell in row:
            cell.number_format = numbers.FORMAT_PERCENTAGE_00
    for row in ws_summary.iter_rows(min_row=2, max_row=ws_summary.max_row, min_col=5, max_col=5):
        for cell in row:
            cell.number_format = numbers.FORMAT_PERCENTAGE_00
    
    # Apply number format to the "Spend" column
    for row in ws_summary.iter_rows(min_row=2, max_row=ws_summary.max_row, min_col=4, max_col=4):
        for cell in row:
            cell.number_format = numbers.FORMAT_NUMBER_COMMA_SEPARATED1  # Apply number formatting with commas

    # Apply color scale for "Count", "Percentage", and "Spend" columns with orange for max and white for min
    color_scale_rule = ColorScaleRule(start_type="min", start_color="FFFFFF", end_type="max", end_color="FFA500")  # Orange for max values
    
    # Apply color scale to "Count", "Percentage", and "Spend" columns
    ws_summary.conditional_formatting.add(f'B2:B{ws_summary.max_row}', color_scale_rule)
    ws_summary.conditional_formatting.add(f'C2:C{ws_summary.max_row}', color_scale_rule)
    ws_summary.conditional_formatting.add(f'D2:D{ws_summary.max_row}', color_scale_rule)
    ws_summary.conditional_formatting.add(f'E2:E{ws_summary.max_row}', color_scale_rule)

    # Format the "Spend Percentage" column as percentage and the "Spend" column as number
    for row in ws_summary.iter_rows(min_row=2, max_row=ws_summary.max_row, min_col=5, max_col=5):
        for cell in row:
            cell.number_format = numbers.FORMAT_PERCENTAGE_00  # Percent format for Spend Percentage
    for row in ws_summary.iter_rows(min_row=2, max_row=ws_summary.max_row, min_col=4, max_col=4):
        for cell in row:
            cell.number_format = numbers.FORMAT_NUMBER_COMMA_SEPARATED1  # Number format for Spend

    # Format the "aggregated spend" column in the Unmatched Data tab as a number
    aggregated_spend_idx = df_unmatched.columns.get_loc("aggregated spend") + 1  # Get column index (1-based)
    for row in ws_data.iter_rows(min_row=2, max_row=ws_data.max_row, min_col=aggregated_spend_idx, max_col=aggregated_spend_idx):
        for cell in row:
            cell.number_format = numbers.FORMAT_NUMBER_COMMA_SEPARATED1  # Number format for aggregated spend
    
    # Apply conditional formatting to the "Category" column based on value
    for row in ws_summary.iter_rows(min_row=2, max_row=ws_summary.max_row, min_col=1, max_col=1):
        cell = row[0]
        if cell.value == "Record Dropped":
            cell.fill = aqua_blue_fill  # Aqua blue for "Record Dropped"
        elif cell.value in ["Supplier missing minimum required fields: supplier_name and address",  
                            "Record must have either a complete address or a combination of street address, city, and country",
                            "The postal code is incorrect for the specified country/state",
                            "Record must have either a complete address or individual fields (like street address, city, etc.), but not both.",
                            "Record is missing values in the mandatory Internal Supplier Id field.",
                            "the supplier name has more than 50% non-English characters",
                            "Record is missing values in the mandatory Company Name field."]:
            cell.fill = light_aqua_blue_fill  # Light aqua blue for these inputs
        elif cell.value == "Multiple Errors":
            cell.fill = aqua_blue_fill  # Dark aqua blue for "Multiple Errors"
        elif cell.value == "No Match Found":
            cell.fill = red_fill  # Red for "No Match Found"
        elif cell.value == "We did not find a match for this record in our database":
            cell.fill = light_red_fill  # Light red for "We did not find a match..."
        elif cell.value == "Record Failed":
            cell.fill = yellow_fill  # Yellow for "Record Failed"
        elif cell.value == "Duplicate Removed":
            cell.fill = bright_green_fill  # Bright green for "Duplicate Removed"
        elif cell.value == "This record was an exact duplicate of another record in your data and was therefore removed.":
            cell.fill = lighter_bright_green_fill  # Lighter bright green for this input
    
    # Apply border to specified ranges in Unmatched Summary
    ranges_to_outline = ['A3:E11', 'A12:E13', 'A14:E14','A15:E16']
    
    for cell_range in ranges_to_outline:
        rows = ws_summary[cell_range]
        for i, row in enumerate(rows):
            for j, cell in enumerate(row):
                # Apply top border to the first row
                if i == 0:
                    cell.border = Border(top=thin, left=cell.border.left, right=cell.border.right, bottom=cell.border.bottom)
                # Apply bottom border to the last row
                if i == len(rows) - 1:
                    cell.border = Border(bottom=thin, left=cell.border.left, right=cell.border.right, top=cell.border.top)
                # Apply left border to the first column
                if j == 0:
                    cell.border = Border(left=thin, top=cell.border.top, bottom=cell.border.bottom, right=cell.border.right)
                # Apply right border to the last column
                if j == len(row) - 1:
                    cell.border = Border(right=thin, top=cell.border.top, bottom=cell.border.bottom, left=cell.border.left)
          

# Save the output file
print(f"Output file '{output_filename}' has been created successfully.")


Output file 'Step 5_SDP Unmatched Refined - 2024-10-17.xlsx' has been created successfully.
